# Electrophysiology: QTc, QT-prolonging drugs, and channel-gene expression

**For:** electrophysiology, critical care, molecular genetics.

**Questions:**
1. What does QTcF look like by sex and age in core-lab-read ECGs?
2. Do people taking a known-risk QT-prolonging drug have a longer QTcF at the same visit, after adjustment?
3. *Alongside:* where are the long-QT / channelopathy genes expressed across 50+ human tissues (GTEx)?

**Requirements:** a workspace with the Project Baseline Health Study (Limited Tier) data collection added (resources `screener`, `crf`, `analysis`, `externallab`, `corelabreads`, `appsurveys`).

**Data:** Baseline core-lab ECG reads (enrollment and Year 2) plus medications, and GTEx v10 median TPM.
GTEx donors are different people from Baseline participants. The two are analyzed side by side, not joined.

In [ ]:
import subprocess
import gzip, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from google.cloud import bigquery, storage

bq = bigquery.Client()

_resolved = {}

def tbl(dataset, table):
    """Look up a workspace dataset by resource name: tbl('crf', 'MH') -> `<project>.crf.MH`."""
    if dataset not in _resolved:
        _resolved[dataset] = subprocess.run(["wb", "resolve", "--name", dataset],
                                            capture_output=True, text=True, check=True).stdout.strip()
    return f"`{_resolved[dataset]}.{table}`"

def sql(q):
    return bq.query(q).to_dataframe()

## 1. QTcF by sex and age at enrollment

In [ ]:
ecg = sql(f"""
SELECT e.USUBJID, e.VISIT, SAFE_CAST(e.QTCF AS FLOAT64) AS qtcf, SAFE_CAST(e.HR AS FLOAT64) AS hr,
       SAFE_CAST(e.QRS AS FLOAT64) AS qrs, d.SEX AS sex, SAFE_CAST(d.age_at_enrollment AS INT64) AS age
FROM {tbl('corelabreads', 'ECG')} e JOIN {tbl('screener', 'DM')} d USING (USUBJID)
WHERE SAFE_CAST(e.QTCF AS FLOAT64) BETWEEN 300 AND 650
""")
ecg["age_band"] = pd.cut(ecg.age, [17, 30, 45, 60, 75, 100], labels=["18–30", "31–45", "46–60", "61–75", "76+"])
ecg["prolonged"] = np.where(ecg.sex == "Male", ecg.qtcf > 450, ecg.qtcf > 460)
enr = ecg[ecg.VISIT == "STUDY_START"]

fig, ax = plt.subplots(figsize=(8, 4))
for i, (sex, color) in enumerate([("Female", "#c0504d"), ("Male", "#3b6ea5")]):
    data = [enr[(enr.sex == sex) & (enr.age_band == b)].qtcf for b in enr.age_band.cat.categories]
    pos = np.arange(len(data)) * 3 + i
    bp = ax.boxplot(data, positions=pos, widths=.8, patch_artist=True, showfliers=False)
    for b in bp["boxes"]:
        b.set_facecolor(color); b.set_alpha(.5)
ax.set_xticks(np.arange(5) * 3 + .5, enr.age_band.cat.categories)
ax.set_ylabel("QTcF (ms)"); ax.set_xlabel("age at enrollment")
ax.legend([plt.Rectangle((0, 0), 1, 1, color=c, alpha=.5) for c in ("#c0504d", "#3b6ea5")], ["Female", "Male"])
ax.set_title("QTcF at enrollment"); ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

(enr.groupby(["sex", "age_band"], observed=True)
    .agg(n=("qtcf", "size"), median_qtcf=("qtcf", "median"), pct_prolonged=("prolonged", lambda s: round(s.mean() * 100, 1))))

## 2. Known-risk QT-prolonging drugs at the same visit (Year 2)
An illustrative list of drugs with *known* torsades risk (CredibleMeds "Known Risk" category). Check the
list against the current CredibleMeds release before any real use. Exposure = drug reported as ongoing at
the Year 2 visit, which is the same visit as the Year 2 ECG.

In [ ]:
QT_DRUGS = ["citalopram", "escitalopram", "ondansetron", "azithromycin", "clarithromycin", "erythromycin",
            "levofloxacin", "ciprofloxacin", "moxifloxacin", "fluconazole", "hydroxychloroquine", "methadone",
            "haloperidol", "amiodarone", "sotalol", "dofetilide", "flecainide", "dronedarone", "donepezil"]
meds = sql(f"""
SELECT USUBJID, LOWER(PrefTerm) AS drug
FROM {tbl('crf', 'CM')}
WHERE VISIT = 'YEAR_TWO_ANNUAL_VISIT' AND CMONGO = 'Y'
  AND REGEXP_CONTAINS(LOWER(PrefTerm), r'{"|".join(QT_DRUGS)}')
""")
print(meds.drug.str.split().str[0].value_counts().head(10).to_string())

y2 = ecg[ecg.VISIT == "YEAR_TWO_ANNUAL_VISIT"].drop_duplicates("USUBJID").copy()
y2["qt_drug"] = y2.USUBJID.isin(meds.USUBJID).astype(int)
print(f"\nYear-2 ECGs: {len(y2):,} · on a known-risk QT drug: {y2.qt_drug.sum()}")

model = smf.ols("qtcf ~ qt_drug + age + C(sex) + hr", data=y2).fit()
print(model.summary().tables[1])

In [ ]:
summary = y2.groupby("qt_drug").agg(n=("qtcf", "size"), mean_qtcf=("qtcf", "mean"),
                                    pct_prolonged=("prolonged", "mean"))
summary["pct_prolonged"] *= 100
coef, (lo, hi) = model.params["qt_drug"], model.conf_int().loc["qt_drug"]
print(f"Adjusted difference in QTcF on a known-risk drug: {coef:+.1f} ms (95% CI {lo:+.1f} to {hi:+.1f})")
summary.rename(index={0: "not exposed", 1: "known-risk QT drug"}).round(1)

## 3. Alongside: where are channelopathy genes expressed? (GTEx v10)
Reads one 9 MB file straight from the GTEx bucket that is linked into this workspace.

In [ ]:
GENES = ["KCNQ1", "KCNH2", "SCN5A", "KCNJ2", "CACNA1C", "KCNE1", "KCNE2", "CALM1", "CALM2", "CALM3",
         "RYR2", "CASQ2", "TRDN", "ANK2", "HCN4"]
blob = storage.Client().bucket("adult-gtex").blob(
    "bulk-gex/v10/rna-seq/GTEx_Analysis_v10_RNASeQCv2.4.2_gene_median_tpm.gct.gz")
gtex = pd.read_csv(io.BytesIO(gzip.decompress(blob.download_as_bytes())), sep="\t", skiprows=2)
expr = gtex[gtex.Description.isin(GENES)].set_index("Description").drop(columns="Name")
expr = expr.loc[[g for g in GENES if g in expr.index]]

TISSUES = ["Heart_Atrial_Appendage", "Heart_Left_Ventricle", "Muscle_Skeletal", "Artery_Aorta", "Artery_Coronary",
           "Esophagus_Muscularis", "Brain_Cortex", "Brain_Cerebellum", "Nerve_Tibial", "Lung", "Liver",
           "Kidney_Cortex", "Adipose_Subcutaneous", "Whole_Blood"]
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(np.log2(expr[TISSUES] + 1), aspect="auto", cmap="viridis")
ax.set_yticks(range(len(expr)), expr.index)
ax.set_xticks(range(len(TISSUES)), [t.replace("_", " ") for t in TISSUES], rotation=60, ha="right", fontsize=8)
plt.colorbar(im, label="log2(median TPM + 1)"); ax.set_title("Channelopathy gene panel: median expression, heart vs. other tissues (GTEx v10)")
plt.tight_layout(); plt.show()

**Try next:** restrict section 2 to participants without structural heart disease (`corelabreads.ECHO`).
Swap `GENES` for a cardiomyopathy panel. For per-donor expression in heart left ventricle, read
`bulk-gex/v10/rna-seq/tpms-by-tissue/gene_tpm_v10_heart_left_ventricle.gct.gz` (see notebook 04).

## Provenance

Generate information about this notebook environment and the packages installed.

In [ ]:
!date

Conda and pip installed packages:

In [ ]:
!conda env export

JupyterLab extensions:

In [ ]:
!jupyter labextension list

Number of cores:

In [ ]:
!grep ^processor /proc/cpuinfo | wc -l

Memory:

In [ ]:
!grep "^MemTotal:" /proc/meminfo

---
Copyright 2026 Verily Life Sciences LLC

Use of this source code is governed by a BSD-style
license that can be found in the LICENSE file or at
https://developers.google.com/open-source/licenses/bsd